# Luna 1 — Dataset Preparation (ChatML conversion)

This notebook is **stage 1** of the Luna 1 fine-tuning pipeline: converting the
raw, human-readable dataset (`nebula_backend.jsonl` and future topic batches)
into Qwen2.5-Coder's ChatML training format.

**What this notebook does today:**
- Pulls the raw dataset(s) from the HuggingFace Hub
- Converts each record into a ChatML string using `render_to_chatml.py`
- Validates the output and shows a couple of rendered samples
- Saves the ChatML dataset locally (and optionally pushes it back to the Hub as a separate `-chatml` split/repo, your choice)

**What this notebook does NOT do (yet):**
- Actual SFT training (LoRA/QLoRA setup, `trl.SFTTrainer`, etc.) — that's a
  separate notebook stage, added once this conversion stage is finalized.

> Why keep raw and ChatML separate instead of just storing ChatML on the Hub?
> The raw schema (`plan`, individual `tool_calls`, `meta.efficiency_label`, ...)
> is the actual source of truth we edit and extend. ChatML is a *rendering* of
> that data for one specific tokenizer/template. Doing the conversion here, in
> a notebook, means there's never a stale/out-of-sync ChatML copy sitting on
> the Hub — you always render fresh from source right before training.

## 1. Setup

In [ ]:
!pip install -q datasets huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
import json
import os

# --- Config: edit these for your repo/paths ---
HF_REPO_ID = "YOUR_USERNAME/luna-dataset"   # <-- set this to your actual HF dataset repo
RAW_FILENAME = "nebula_backend.jsonl"        # topic file to convert; loop over multiple later as more batches are added
LOCAL_DIR = "/content/luna_dataset"

os.makedirs(LOCAL_DIR, exist_ok=True)

## 2. Pull the raw dataset + conversion script from the Hub

In [ ]:
raw_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=RAW_FILENAME,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
)

script_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="render_to_chatml.py",
    repo_type="dataset",
    local_dir=LOCAL_DIR,
)

print("Raw dataset:", raw_path)
print("Conversion script:", script_path)

## 3. Load the conversion logic

This is the exact same `render_to_chatml.py` shipped in the dataset repo —
imported directly rather than re-implemented here, so the notebook can never
drift out of sync with the canonical conversion logic.

In [ ]:
import sys
sys.path.insert(0, LOCAL_DIR)

from render_to_chatml import render_record, convert_file

print("Loaded render_record() and convert_file() from render_to_chatml.py")

## 4. Convert raw → ChatML

In [ ]:
chatml_path = os.path.join(LOCAL_DIR, RAW_FILENAME.replace(".jsonl", "_chatml.jsonl"))

convert_file(raw_path, chatml_path)

## 5. Sanity-check the output

In [ ]:
with open(chatml_path, encoding="utf-8") as f:
    lines = [json.loads(l) for l in f if l.strip()]

print(f"{len(lines)} records converted.\n")

# Show one full rendered example
sample = lines[0]
print("=" * 80)
print(f"efficiency_label: {sample['meta']['efficiency_label']}  |  tool_calls: {sample['meta']['num_tool_calls']}")
print("=" * 80)
print(sample["text"])

In [ ]:
# Quick structural checks before this is considered training-ready
required_tags_present = all(
    "<|im_start|>" in rec["text"] and "<|im_end|>" in rec["text"]
    for rec in lines
)
print("All records contain ChatML tags:", required_tags_present)

# Rough token-length sanity check (character count as a cheap proxy pre-tokenizer)
lengths = [len(rec["text"]) for rec in lines]
print(f"Char length -- min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}")

## 6. Verify against the actual Qwen2.5-Coder tokenizer

`render_to_chatml.py` renders `tool` messages as `user`-role turns wrapped in
`<tool_response>` tags, since this is the common convention for chat templates
without a dedicated `tool` role. **Verify this matches the real
`tokenizer_config.json`** for whichever Qwen2.5-Coder checkpoint you're
targeting before training — some checkpoints do support a native tool role.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # <-- swap for the exact checkpoint you're fine-tuning

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Does this checkpoint's chat template define a native 'tool' role?
has_native_tool_role = "tool" in (tokenizer.chat_template or "")
print(f"{MODEL_ID} chat_template mentions a native tool role:", has_native_tool_role)
print()
print("If True: consider remapping our <tool_response>-wrapped user turns to")
print("the native tool role before training, for template fidelity.")
print("If False: our current user-role wrapping convention is a safe default.")

In [ ]:
# Tokenize one sample end-to-end as a final sanity pass
tokens = tokenizer(sample["text"])["input_ids"]
print(f"Sample record tokenizes to {len(tokens)} tokens.")
print("Decoded back (first 300 chars):")
print(tokenizer.decode(tokens)[:300])

## 7. Save locally

The ChatML file is now saved at `chatml_path`, ready to be consumed directly
by an SFT trainer in the next notebook stage
(`datasets.load_dataset("json", data_files=chatml_path)`).

Pushing this ChatML output back to the Hub is optional and deliberately not
done automatically here — see the note at the top of this notebook on why we
treat ChatML as a derived, re-renderable artifact rather than a second
source of truth. If you do want a Hub copy of a specific ChatML render (e.g.
to share alongside a specific model release), uncomment and run the cell
below.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=chatml_path,
#     path_in_repo=os.path.basename(chatml_path),
#     repo_id=HF_REPO_ID,
#     repo_type="dataset",
# )
# print("Uploaded:", chatml_path)

---

## Next stage (not in this notebook yet)

Once this conversion step is finalized across all topic batches, the next
notebook stage will add:
- LoRA/QLoRA config for Qwen2.5-Coder (small footprint, fits Colab's free-tier GPU)
- `trl.SFTTrainer` setup consuming this ChatML output directly
- Eval harness comparing `optimal` vs `wasteful`-labeled completions

This will be added as `02_train_luna.ipynb` once the dataset has enough
coverage across task types to be worth a first training run.